# Exploración Extra: Encoding Avanzado - Employee Attrition Dataset
## Autor: Valentín Rodríguez
**UT3: Feature Engineering | Comparación de Técnicas de Encoding con Datos de RR.HH.**

## 🎯 Objetivos
- Comparar diferentes técnicas de encoding categórico en un dataset real de recursos humanos
- Implementar Target Encoding con prevención de data leakage usando cross-validation
- Crear pipelines con branching usando ColumnTransformer
- Analizar trade-offs entre accuracy, dimensionalidad y tiempo de entrenamiento

## 📂 Dataset: Employee Attrition
**Datos de recursos humanos con patrones de rotación de personal**

### Características:
- **1,470 empleados** con información completa
- **Variables categóricas**: Department, JobRole, EducationField, BusinessTravel, etc.
- **Target**: Attrition (Yes/No) - ¿El empleado dejó la empresa?


In [ ]:
%pip install category-encoders seaborn scikit-learn matplotlib pandas numpy --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from category_encoders import TargetEncoder
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8')
sns.set_palette("Set2")

print("✅ Entorno configurado")

## Cargar y Explorar Dataset

In [ ]:
# Generar datos sintéticos basados en IBM HR Analytics
np.random.seed(42)
n_samples = 1470

departments = ['Sales', 'Research & Development', 'Human Resources']
job_roles = ['Sales Executive', 'Research Scientist', 'Laboratory Technician',
             'Manufacturing Director', 'Healthcare Representative', 'Manager',
             'Sales Representative', 'Research Director', 'Human Resources']
education_fields = ['Life Sciences', 'Medical', 'Marketing', 'Technical Degree',
                    'Other', 'Human Resources']
business_travel = ['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
marital_status = ['Single', 'Married', 'Divorced']
overtime = ['Yes', 'No']

data = {
    'Age': np.random.randint(18, 65, n_samples),
    'Department': np.random.choice(departments, n_samples),
    'Education': np.random.randint(1, 5, n_samples),
    'EducationField': np.random.choice(education_fields, n_samples),
    'JobRole': np.random.choice(job_roles, n_samples),
    'BusinessTravel': np.random.choice(business_travel, n_samples),
    'MaritalStatus': np.random.choice(marital_status, n_samples),
    'MonthlyIncome': np.random.randint(2000, 20000, n_samples),
    'YearsAtCompany': np.random.randint(0, 40, n_samples),
    'OverTime': np.random.choice(overtime, n_samples),
    'WorkLifeBalance': np.random.randint(1, 4, n_samples),
    'JobSatisfaction': np.random.randint(1, 4, n_samples),
}

df = pd.DataFrame(data)

# Crear target con relación a las variables
attrition_prob = (
    (df['JobSatisfaction'] < 2) * 0.4 +
    (df['WorkLifeBalance'] < 2) * 0.3 +
    (df['OverTime'] == 'Yes') * 0.2 +
    (df['Department'] == 'Sales') * 0.1 +
    np.random.rand(n_samples) * 0.1
)
attrition_prob = np.clip(attrition_prob, 0, 1)
df['Attrition'] = (np.random.rand(n_samples) < attrition_prob).astype(int)
df['Attrition'] = df['Attrition'].map({0: 'No', 1: 'Yes'})

print(f"Dataset shape: {df.shape}")
print(f"Target distribution: {df['Attrition'].value_counts()}")

## Análisis de Cardinalidad

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'Attrition' in categorical_cols:
    categorical_cols.remove('Attrition')

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("🔍 ANÁLISIS DETALLADO DE CARDINALIDAD")
print("=" * 80)

# Calcular cardinalidad
cardinality_data = []
for col in categorical_cols:
    n_unique = df[col].nunique()
    cardinality_type = 'Baja' if n_unique <= 10 else ('Media' if n_unique <= 50 else 'Alta')
    cardinality_data.append({
        'Variable': col,
        'Cardinalidad': n_unique,
        'Tipo': cardinality_type
    })

cardinality_df = pd.DataFrame(cardinality_data).sort_values('Cardinalidad', ascending=False)

print("\n📊 TABLA DE CARDINALIDAD:")
print(cardinality_df.to_string(index=False))

# Clasificar variables
low_card_cols = [col for col in categorical_cols if df[col].nunique() <= 10]
high_card_cols = [col for col in categorical_cols if df[col].nunique() > 10]

print(f"\n Variables de BAJA cardinalidad (≤10): {len(low_card_cols)}")
print(f"   {low_card_cols}")

print(f"\n Variables de ALTA/MEDIA cardinalidad (>10): {len(high_card_cols)}")
print(f"   {high_card_cols}")

# Visualización de cardinalidad
plt.figure(figsize=(12, 6))
colors = ['#2ecc71' if x <= 10 else '#e74c3c' for x in cardinality_df['Cardinalidad']]
plt.barh(cardinality_df['Variable'], cardinality_df['Cardinalidad'], color=colors, alpha=0.7)
plt.xlabel('Cardinalidad (Número de Categorías Únicas)', fontsize=12)
plt.ylabel('Variable Categórica', fontsize=12)
plt.title('Análisis de Cardinalidad de Variables Categóricas\n(Verde: Baja ≤10, Rojo: Alta >10)', fontsize=14)
plt.axvline(x=10, color='orange', linestyle='--', linewidth=2, label='Umbral Baja/Alta Cardinalidad')
plt.legend()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../assets/employee-attrition-cardinality.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n💡 INSIGHTS:")
print(f"   - Total one-hot columns si usáramos todo: {sum([df[col].nunique() for col in categorical_cols])}")
print(f"   - Variables originales: {len(categorical_cols)}")
print(f"   - Expansión dimensional: {sum([df[col].nunique() for col in categorical_cols]) / len(categorical_cols):.1f}x")
print(f"   - Recomendación: Usar One-Hot solo para baja cardinalidad, Target Encoding para alta")

## Paso 3: Preparar Datos para Experimentación


In [ ]:
# === PREPARAR DATOS ===

print("\n🔄 PREPARANDO DATOS PARA EXPERIMENTACIÓN")
print("=" * 80)

# Crear target binario
y = (df['Attrition'] == 'Yes').astype(int)
X = df.drop('Attrition', axis=1)

# Separar categóricas y numéricas
X_cat = X[categorical_cols].copy()
X_num = X[numerical_cols].copy()

# Train/Test split
X_train_cat, X_test_cat, X_train_num, X_test_num, y_train, y_test = train_test_split(
    X_cat, X_num, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✅ Datos preparados:")
print(f"   Train: {X_train_cat.shape[0]} muestras")
print(f"   Test: {X_test_cat.shape[0]} muestras")
print(f"   Distribución train - Attrition: {y_train.mean():.1%}")
print(f"   Distribución test - Attrition: {y_test.mean():.1%}")


## 🔬 Paso 4: Experimentos de Encoding

### 4.1 Label Encoding


In [ ]:
# === LABEL ENCODING ===

print("\n🔢 EXPERIMENTO 1: LABEL ENCODING")
print("=" * 80)

start_time = time.time()

X_train_label = X_train_cat.copy()
X_test_label = X_test_cat.copy()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train_label[col] = le.fit_transform(X_train_label[col])
    label_encoders[col] = le
    
    # Manejar categorías no vistas
    le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    X_test_label[col] = X_test_cat[col].map(le_dict).fillna(-1).astype(int)

# Combinar con numéricas
X_train_encoded = pd.concat([X_train_num.reset_index(drop=True), X_train_label.reset_index(drop=True)], axis=1)
X_test_encoded = pd.concat([X_test_num.reset_index(drop=True), X_test_label.reset_index(drop=True)], axis=1)

# Entrenar modelo
rf_label = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_label.fit(X_train_encoded, y_train)

y_pred_label = rf_label.predict(X_test_encoded)
y_pred_proba_label = rf_label.predict_proba(X_test_encoded)[:, 1]

accuracy_label = accuracy_score(y_test, y_pred_label)
auc_label = roc_auc_score(y_test, y_pred_proba_label)
f1_label = f1_score(y_test, y_pred_label)

elapsed_time = time.time() - start_time

print(f"\n RESULTADOS LABEL ENCODING:")
print(f"   Accuracy: {accuracy_label:.4f} ({accuracy_label*100:.2f}%)")
print(f"   AUC-ROC: {auc_label:.4f} ({auc_label*100:.2f}%)")
print(f"   F1-Score: {f1_label:.4f} ({f1_label*100:.2f}%)")
print(f"   Features: {X_train_encoded.shape[1]}")
print(f"   Tiempo: {elapsed_time:.2f}s")


### 4.2 One-Hot Encoding (solo baja cardinalidad)


In [ ]:
# === ONE-HOT ENCODING (SOLO BAJA CARDINALIDAD) ===

print("\n🔥 EXPERIMENTO 2: ONE-HOT ENCODING (SOLO BAJA CARDINALIDAD)")
print("=" * 80)

start_time = time.time()

# One-Hot solo para baja cardinalidad
if len(low_card_cols) > 0:
    encoder_ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    X_train_ohe = encoder_ohe.fit_transform(X_train_cat[low_card_cols])
    X_test_ohe = encoder_ohe.transform(X_test_cat[low_card_cols])
    
    # Label encoding para alta cardinalidad
    X_train_high_label = X_train_cat[high_card_cols].copy()
    X_test_high_label = X_test_cat[high_card_cols].copy()
    
    for col in high_card_cols:
        le = LabelEncoder()
        X_train_high_label[col] = le.fit_transform(X_train_high_label[col])
        le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
        X_test_high_label[col] = X_test_cat[col].map(le_dict).fillna(-1).astype(int)
    
    # Combinar
    X_train_ohe_combined = pd.concat([
        pd.DataFrame(X_train_ohe, columns=[f'ohe_{i}' for i in range(X_train_ohe.shape[1])]),
        X_train_high_label.reset_index(drop=True),
        X_train_num.reset_index(drop=True)
    ], axis=1)
    
    X_test_ohe_combined = pd.concat([
        pd.DataFrame(X_test_ohe, columns=[f'ohe_{i}' for i in range(X_test_ohe.shape[1])]),
        X_test_high_label.reset_index(drop=True),
        X_test_num.reset_index(drop=True)
    ], axis=1)
else:
    X_train_ohe_combined = X_train_num.reset_index(drop=True)
    X_test_ohe_combined = X_test_num.reset_index(drop=True)

# Entrenar modelo
rf_ohe = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_ohe.fit(X_train_ohe_combined, y_train)

y_pred_ohe = rf_ohe.predict(X_test_ohe_combined)
y_pred_proba_ohe = rf_ohe.predict_proba(X_test_ohe_combined)[:, 1]

accuracy_ohe = accuracy_score(y_test, y_pred_ohe)
auc_ohe = roc_auc_score(y_test, y_pred_proba_ohe)
f1_ohe = f1_score(y_test, y_pred_ohe)

elapsed_time = time.time() - start_time

print(f"\n✅ RESULTADOS ONE-HOT ENCODING:")
print(f"   Accuracy: {accuracy_ohe:.4f} ({accuracy_ohe*100:.2f}%)")
print(f"   AUC-ROC: {auc_ohe:.4f} ({auc_ohe*100:.2f}%)")
print(f"   F1-Score: {f1_ohe:.4f} ({f1_ohe*100:.2f}%)")
print(f"   Features: {X_train_ohe_combined.shape[1]}")
print(f"   Tiempo: {elapsed_time:.2f}s")


### 4.3 Target Encoding (alta cardinalidad)


In [ ]:
# === TARGET ENCODING ===

print("\n🎯 EXPERIMENTO 3: TARGET ENCODING")
print("=" * 80)

start_time = time.time()

if len(high_card_cols) > 0:
    # Target encoding para alta cardinalidad
    encoder_target = TargetEncoder(cols=high_card_cols, smoothing=10.0)
    X_train_target = encoder_target.fit_transform(X_train_cat[high_card_cols], y_train)
    X_test_target = encoder_target.transform(X_test_cat[high_card_cols])
    
    # One-Hot para baja cardinalidad
    if len(low_card_cols) > 0:
        encoder_ohe_target = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
        X_train_target_low = encoder_ohe_target.fit_transform(X_train_cat[low_card_cols])
        X_test_target_low = encoder_ohe_target.transform(X_test_cat[low_card_cols])
        
        X_train_target_combined = pd.concat([
            pd.DataFrame(X_train_target_low, columns=[f'te_ohe_{i}' for i in range(X_train_target_low.shape[1])]),
            X_train_target.reset_index(drop=True),
            X_train_num.reset_index(drop=True)
        ], axis=1)
        
        X_test_target_combined = pd.concat([
            pd.DataFrame(X_test_target_low, columns=[f'te_ohe_{i}' for i in range(X_test_target_low.shape[1])]),
            X_test_target.reset_index(drop=True),
            X_test_num.reset_index(drop=True)
        ], axis=1)
    else:
        X_train_target_combined = pd.concat([
            X_train_target.reset_index(drop=True),
            X_train_num.reset_index(drop=True)
        ], axis=1)
        
        X_test_target_combined = pd.concat([
            X_test_target.reset_index(drop=True),
            X_test_num.reset_index(drop=True)
        ], axis=1)
else:
    X_train_target_combined = X_train_num.reset_index(drop=True)
    X_test_target_combined = X_test_num.reset_index(drop=True)

# Entrenar modelo
rf_target = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_target.fit(X_train_target_combined, y_train)

y_pred_target = rf_target.predict(X_test_target_combined)
y_pred_proba_target = rf_target.predict_proba(X_test_target_combined)[:, 1]

accuracy_target = accuracy_score(y_test, y_pred_target)
auc_target = roc_auc_score(y_test, y_pred_proba_target)
f1_target = f1_score(y_test, y_pred_target)

elapsed_time = time.time() - start_time

print(f"\n✅ RESULTADOS TARGET ENCODING:")
print(f"   Accuracy: {accuracy_target:.4f} ({accuracy_target*100:.2f}%)")
print(f"   AUC-ROC: {auc_target:.4f} ({auc_target*100:.2f}%)")
print(f"   F1-Score: {f1_target:.4f} ({f1_target*100:.2f}%)")
print(f"   Features: {X_train_target_combined.shape[1]}")
print(f"   Tiempo: {elapsed_time:.2f}s")


## Paso 5: Comparación de Resultados y Visualización

In [ ]:
# === COMPARACIÓN FINAL ===

print("\n📊 COMPARACIÓN FINAL DE MÉTODOS")
print("=" * 80)

# Crear DataFrame de resultados
results_data = {
    'Método': ['Label Encoding', 'One-Hot (low card)', 'Target Encoding'],
    'Accuracy': [accuracy_label, accuracy_ohe, accuracy_target],
    'AUC-ROC': [auc_label, auc_ohe, auc_target],
    'F1-Score': [f1_label, f1_ohe, f1_target]
}

results_df = pd.DataFrame(results_data)

print("\n📋 TABLA COMPARATIVA:")
print(results_df.to_string(index=False))

# Visualización comparativa
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(results_df['Método'], results_df['Accuracy'], color=['#3498db', '#e74c3c', '#2ecc71'], alpha=0.7)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Comparación de Accuracy', fontsize=14)
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(results_df['Accuracy']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

axes[1].bar(results_df['Método'], results_df['AUC-ROC'], color=['#3498db', '#e74c3c', '#2ecc71'], alpha=0.7)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title('Comparación de AUC-ROC', fontsize=14)
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(results_df['AUC-ROC']):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

axes[2].bar(results_df['Método'], results_df['F1-Score'], color=['#3498db', '#e74c3c', '#2ecc71'], alpha=0.7)
axes[2].set_ylabel('F1-Score', fontsize=12)
axes[2].set_title('Comparación de F1-Score', fontsize=14)
axes[2].set_ylim([0, 1])
axes[2].grid(axis='y', alpha=0.3)
for i, v in enumerate(results_df['F1-Score']):
    axes[2].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../assets/employee-attrition-encoding-comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n🏆 MEJOR MÉTODO POR MÉTRICA:")
print(f"   Accuracy: {results_df.loc[results_df['Accuracy'].idxmax(), 'Método']} ({results_df['Accuracy'].max():.4f})")
print(f"   AUC-ROC: {results_df.loc[results_df['AUC-ROC'].idxmax(), 'Método']} ({results_df['AUC-ROC'].max():.4f})")
print(f"   F1-Score: {results_df.loc[results_df['F1-Score'].idxmax(), 'Método']} ({results_df['F1-Score'].max():.4f})")

print("\n💡 INSIGHTS:")
print("   - Comparar rendimiento de diferentes técnicas de encoding")
print("   - Analizar trade-offs entre dimensionalidad y accuracy")
print("   - Considerar tiempo de entrenamiento para producción")
print("\n✅ ANÁLISIS DE ENCODING AVANZADO COMPLETADO")
print("=" * 80)
